# Sports Player Tracking Pipeline (Google Colab Notebook)

End-to-end notebook for tracking players in sports videos:

1. **Setup** — install dependencies and configure paths
2. **YOLO v11** — player detection
3. **OpenPose** — body keypoint estimation
4. **ByteTrack** — multi-frame player tracking
5. **Full pipeline** — process all videos and save results
6. **Evaluation** — accuracy, precision, recall, F1, and loss curves

---
## Part 1: Setup & Install Dependencies

In [ ]:
# Check GPU availability (recommended for YOLO + OpenPose)
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies
!pip install -q ultralytics opencv-python controlnet-aux PyYAML tqdm matplotlib lap pandas

In [ ]:
# Mount Google Drive 
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/DS5216-PA2'  
%cd {PROJECT_ROOT}

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

# Add project root so "from src ..." imports work
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config_loader import load_config, get_video_paths

config = load_config()

# Paths are inside the project folder: SportsVideos/ and outputs/
videos = get_video_paths(config)
print(f"Project root: {PROJECT_ROOT}")
print(f"Video dir:    {config['paths']['video_dir']}")
print(f"Output dir:   {config['paths']['output_dir']}")
print(f"Found {len(videos)} video(s):")
for v in videos:
    print(f"  - {v.name}")

In [ ]:
# Load first frame (reused in Parts 2 and 3)
if not videos:
    raise FileNotFoundError("No videos found. Check video_dir path in config.")

cap = cv2.VideoCapture(str(videos[0]))
ret, frame = cap.read()
cap.release()

if ret:
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(videos[0].name)
    plt.axis('off')
    plt.show()
    print(f"Frame shape: {frame.shape}")
else:
    raise RuntimeError(f"Could not read frame from {videos[0]}")

---
## Part 2: Player Detection with YOLO v11

In [ ]:
from src.detection.yolo_detector import YOLOPlayerDetector
from src.utils.visualization import draw_bounding_box

yolo_cfg = config['models']['yolo']
detector = YOLOPlayerDetector(
    model_name=yolo_cfg['model_name'],
    confidence=yolo_cfg['confidence'],
    iou=yolo_cfg['iou'],
    person_class_id=yolo_cfg['person_class_id'],
)

detections = detector.detect(frame)
print(f"Detected {len(detections)} player(s) in {videos[0].name}")
for i, det in enumerate(detections):
    print(f"  Player {i + 1}: bbox={det.bbox}, confidence={det.confidence:.2f}")

In [ ]:
vis = frame.copy()
for det in detections:
    vis = draw_bounding_box(
        vis, det.bbox, f"Player ({det.confidence:.2f})",
        color=(0, 255, 0), thickness=2,
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"YOLO v11 Detection - {len(detections)} players")
plt.axis('off')
plt.show()

---
## Part 3: Keypoint Detection with OpenPose

In [ ]:
from src.keypoints.openpose_estimator import OpenPoseEstimator, BODY_JOINT_NAMES
from src.utils.visualization import draw_keypoints

if not detections:
    raise ValueError("No players detected. Try lowering confidence threshold in config.yaml.")

bbox = detections[0].bbox
print(f"Using player bbox: {bbox}")

openpose_cfg = config['models']['openpose']
pose_estimator = OpenPoseEstimator(
    device=openpose_cfg['device'],
    include_hands=openpose_cfg['include_hands'],
    include_face=openpose_cfg['include_face'],
)

keypoints = pose_estimator.estimate_on_crop(frame, bbox)

if keypoints is not None:
    visible = sum(1 for kp in keypoints if kp[2] > 0.1)
    print(f"Detected {visible} visible keypoints out of 18")
    for name, (x, y, conf) in zip(BODY_JOINT_NAMES, keypoints):
        if conf > 0.1:
            print(f"  {name}: ({x:.0f}, {y:.0f}) conf={conf:.2f}")
else:
    print("No pose detected in crop.")

In [ ]:
vis = frame.copy()
if keypoints is not None:
    vis = draw_keypoints(vis, keypoints, (0, 0, 255), (255, 128, 0))

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("OpenPose Keypoints on Player")
plt.axis('off')
plt.show()

---
## Part 4: Player Tracking with ByteTrack

In [ ]:
from src.tracking.player_tracker import PlayerTracker

track_cfg = config['models']['tracking']
tracker = PlayerTracker(
    model_name=yolo_cfg['model_name'],
    confidence=yolo_cfg['confidence'],
    tracker=track_cfg['tracker'],
    persist=track_cfg['persist'],
)

NUM_FRAMES = 30
cap = cv2.VideoCapture(str(videos[0]))
track_history = {}

for frame_idx in range(NUM_FRAMES):
    ret, track_frame = cap.read()
    if not ret:
        break

    tracked = tracker.track(track_frame)
    for player in tracked:
        track_history.setdefault(player.track_id, []).append(
            (frame_idx, player.bbox)
        )

cap.release()

print(f"Tracked {len(track_history)} unique player(s) over {NUM_FRAMES} frames:")
for tid, history in track_history.items():
    print(f"  Player ID {tid}: seen in {len(history)} frames")

In [ ]:
# Visualize tracking on frame 15
cap = cv2.VideoCapture(str(videos[0]))
for _ in range(15):
    cap.read()
ret, track_frame = cap.read()
cap.release()

tracked = tracker.track(track_frame)
vis = track_frame.copy()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

for player in tracked:
    color = colors[player.track_id % len(colors)]
    vis = draw_bounding_box(
        vis, player.bbox, f"ID {player.track_id}", color, thickness=2
    )

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"ByteTrack - {len(tracked)} players tracked")
plt.axis('off')
plt.show()

---
## Part 5: Full Pipeline — Track All Sports Videos

In [ ]:
from src.pipeline.tracking_pipeline import PlayerTrackingPipeline

# Quick test: 50 frames per video (set to None for full video)
config['processing']['max_frames'] = 50
config['processing']['save_video'] = True
config['processing']['save_json'] = True

print(f"Will process {len(videos)} video(s):")
for v in videos:
    print(f"  - {v.name}")

In [ ]:
pipeline = PlayerTrackingPipeline(config)
results = pipeline.process_all_videos(videos)

In [ ]:
for result in results:
    video_name = Path(result['video']).name
    num_frames = len(result['frames'])
    total_players = sum(len(f['players']) for f in result['frames'])
    print(f"{video_name}: {num_frames} frames, {total_players} total player detections")

In [ ]:
from IPython.display import Video, display

output_dir = Path(config['paths']['output_dir']) / videos[0].stem
output_video = output_dir / f"{videos[0].stem}_tracked.mp4"

if output_video.exists():
    display(Video(str(output_video), width=640))
else:
    print(f"Output not found: {output_video}")

In [ ]:
# Process ALL frames on ALL videos 
config['processing']['max_frames'] = None
pipeline = PlayerTrackingPipeline(config)
results = pipeline.process_all_videos(videos)

---
## Part 6: Evaluation — Accuracy, Precision, Recall, F1 & Loss Curves

This section evaluates player detection in two ways:

1. **Sports video evaluation** — compare YOLO v11n predictions against proxy ground truth (from YOLO v11m) on sample frames
2. **YOLO validation & training** — official COCO8 metrics and loss curves from a short training run

> For a real project, replace proxy annotations with human-labeled JSON files in `data/annotations/`.

In [ ]:
import sys
from pathlib import Path

# Run this cell first if you get: ModuleNotFoundError: No module named 'src'
PROJECT_ROOT = '/content/drive/MyDrive/DS5216-PA2'  # <-- update to match your Drive path
%cd {PROJECT_ROOT}

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if not Path('src/evaluation/metrics.py').exists():
    raise FileNotFoundError(
        f"Could not find src/ in {PROJECT_ROOT}.\n"
        "Upload the full project folder to Drive, or git pull the latest code."
    )

from src.config_loader import load_config, get_video_paths
from src.detection.yolo_detector import YOLOPlayerDetector

config = load_config()
videos = get_video_paths(config)
yolo_cfg = config['models']['yolo']
detector = YOLOPlayerDetector(
    model_name=yolo_cfg['model_name'],
    confidence=yolo_cfg['confidence'],
    iou=yolo_cfg['iou'],
    person_class_id=yolo_cfg['person_class_id'],
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Video dir:    {config['paths']['video_dir']}")
print(f"Output dir:   {config['paths']['output_dir']}")
print(f"Videos found: {len(videos)}")

In [ ]:
from src.evaluation.metrics import (
    evaluate_at_confidence_thresholds,
    evaluate_video_detections,
    generate_proxy_annotations,
    save_annotations,
)
from src.evaluation.yolo_eval import (
    plot_detection_metrics,
    plot_loss_curves,
    plot_precision_recall_curve,
    run_yolo_training,
    run_yolo_validation,
)

# Sample frames to evaluate (evenly spaced)
EVAL_FRAME_INDICES = [0, 50, 100, 150, 200, 250, 290]

# Create proxy ground truth using a stronger YOLO model
teacher_detector = YOLOPlayerDetector(model_name="yolo11m.pt", confidence=0.5)
proxy_annotations = generate_proxy_annotations(
    video_path=videos[0],
    detector=teacher_detector,
    frame_indices=EVAL_FRAME_INDICES,
)

annotation_path = Path(config["paths"]["output_dir"]) / "evaluation" / f"{videos[0].stem}_proxy_annotations.json"
save_annotations(proxy_annotations, annotation_path, videos[0].name)
print(f"Proxy annotations saved to: {annotation_path}")
print(f"Annotated frames: {len(proxy_annotations)}")

In [ ]:
# Evaluate YOLO v11n against proxy ground truth
sports_metrics = evaluate_video_detections(
    detector=detector,
    video_path=videos[0],
    annotations=proxy_annotations,
    iou_threshold=0.5,
)

print("=== Sports Video Detection Metrics (YOLO v11n) ===")
print(f"  Precision:  {sports_metrics['precision']:.4f}")
print(f"  Recall:     {sports_metrics['recall']:.4f}")
print(f"  F1 Score:   {sports_metrics['f1']:.4f}")
print(f"  Accuracy:   {sports_metrics['accuracy']:.4f}")
print(f"  TP / FP / FN / TN: {sports_metrics['tp']} / {sports_metrics['fp']} / {sports_metrics['fn']} / {sports_metrics['tn']}")

plot_detection_metrics(
    sports_metrics,
    title="Sports Video Detection Metrics (YOLO v11n vs proxy GT)",
    save_path=Path(config["paths"]["output_dir"]) / "evaluation" / "sports_metrics.png",
)

In [ ]:
# Precision / Recall / F1 at different confidence thresholds
CONFIDENCE_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = evaluate_at_confidence_thresholds(
    detector=detector,
    video_path=videos[0],
    annotations=proxy_annotations,
    thresholds=CONFIDENCE_THRESHOLDS,
    iou_threshold=0.5,
)

print("=== Metrics vs Confidence Threshold ===")
for row in threshold_results:
    print(
        f"  conf={row['confidence_threshold']:.1f}  "
        f"P={row['precision']:.3f}  R={row['recall']:.3f}  F1={row['f1']:.3f}"
    )

plot_precision_recall_curve(threshold_results)

In [ ]:
# YOLO validation on COCO8 (person class) — standard benchmark metrics
coco_metrics = run_yolo_validation(
    model_name=yolo_cfg["model_name"],
    data_yaml="coco8.yaml",
    person_class_id=yolo_cfg["person_class_id"],
)

print("=== COCO8 Validation Metrics (Person Class) ===")
print(f"  Model:      {coco_metrics['model_name']}")
print(f"  Precision:  {coco_metrics['precision']:.4f}")
print(f"  Recall:     {coco_metrics['recall']:.4f}")
print(f"  F1 Score:   {coco_metrics['f1']:.4f}")
print(f"  mAP@0.5:    {coco_metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {coco_metrics['map50_95']:.4f}")

plot_detection_metrics(
    coco_metrics,
    title="COCO8 Validation Metrics (Person Class)",
    save_path=Path(config["paths"]["output_dir"]) / "evaluation" / "coco_metrics.png",
)

In [ ]:
# Short YOLO training run to generate loss curves (3 epochs for Colab speed)
TRAIN_EPOCHS = 3

run_dir = run_yolo_training(
    model_name=yolo_cfg["model_name"],
    data_yaml="coco8.yaml",
    epochs=TRAIN_EPOCHS,
    project=str(Path(config["paths"]["output_dir"]) / "evaluation"),
    name="yolo_loss_curves",
)

results_csv = run_dir / "results.csv"
print(f"Training complete. Results saved to: {results_csv}")

plot_loss_curves(
    results_csv,
    save_path=Path(config["paths"]["output_dir"]) / "evaluation" / "loss_curves.png",
)